# 🚀 Ferni Tool Router - Qwen 2.5 0.5B (v2 - Extended Training)

Train a fast tool classifier on 45K+ examples with **5 epochs** for production-ready accuracy.

**Improvements over v1:**
- 5 epochs instead of 1
- Learning rate warmup
- Better evaluation logging
- Gradient accumulation for effective larger batch

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets accelerate peft bitsandbytes safetensors torch

In [ ]:
# Cell 2: Upload training files
from google.colab import files
print("📁 Upload these 3 files:")
print("   1. train.jsonl (8 MB)")
print("   2. validation.jsonl (722 KB)")
print("   3. label_map.json (28 KB)")
print("")
uploaded = files.upload()

In [ ]:
# Cell 3: Check GPU
import torch
print(f"🎮 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"💾 Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "❌ No GPU! Go to Runtime > Change runtime type > T4 GPU"

In [ ]:
# Cell 4: Load training data
import json

# Load label map
with open('label_map.json') as f:
    label_map = json.load(f)
num_labels = len(label_map)
print(f"📊 {num_labels} tool labels")

# Load training data
train_data = []
with open('train.jsonl') as f:
    for line in f:
        train_data.append(json.loads(line))
print(f"📚 {len(train_data)} training examples")

# Load validation data
val_data = []
with open('validation.jsonl') as f:
    for line in f:
        val_data.append(json.loads(line))
print(f"✅ {len(val_data)} validation examples")

In [ ]:
# Cell 5: Load model and tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

print(f"📥 Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id

# LoRA config for efficient training
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["classifier", "score"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ Model ready!")

In [ ]:
# Cell 6: Create datasets
from torch.utils.data import Dataset, DataLoader
import numpy as np

class ToolDataset(Dataset):
    def __init__(self, data, tokenizer, label_map, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.num_labels = len(label_map)
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        query = item['query']
        tools = item.get('selected_tools', item.get('tools', []))
        if isinstance(tools, str):
            tools = [tools]

        encoding = self.tokenizer(
            query,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        # Multi-label binary vector
        labels = np.zeros(self.num_labels, dtype=np.float32)
        for tool in tools:
            if tool in self.label_map:
                labels[self.label_map[tool]] = 1.0

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(labels)
        }

train_dataset = ToolDataset(train_data, tokenizer, label_map)
val_dataset = ToolDataset(val_data, tokenizer, label_map)

# DataLoaders with gradient accumulation
BATCH_SIZE = 16  # Smaller batch, accumulate gradients
GRAD_ACCUM = 4   # Effective batch = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f"📊 Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"📊 Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# Cell 7: Training with 5 epochs
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from tqdm.auto import tqdm
import torch.nn as nn

EPOCHS = 5
LR = 2e-4

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM
scheduler = OneCycleLR(optimizer, max_lr=LR, total_steps=total_steps, pct_start=0.1)

loss_fn = nn.BCEWithLogitsLoss()
device = torch.device('cuda')
model.to(device)

print(f"🚀 Training for {EPOCHS} epochs...")
print(f"📊 Total steps: {total_steps}")

best_val_acc = 0
history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels) / GRAD_ACCUM
        loss.backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix({'loss': f"{total_loss/(step+1):.4f}"})

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = (torch.sigmoid(outputs.logits) > 0.5).float()
            
            # Check if top prediction matches any true label
            top_pred = outputs.logits.argmax(dim=1)
            for i in range(len(top_pred)):
                if labels[i, top_pred[i]] == 1.0:
                    val_correct += 1
                val_total += 1

    val_acc = val_correct / val_total
    avg_loss = total_loss / len(train_loader)
    history.append({'epoch': epoch+1, 'loss': avg_loss, 'val_acc': val_acc})
    
    print(f"\n📊 Epoch {epoch+1}: Loss={avg_loss:.4f}, Val Acc={val_acc:.1%}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained('output/best')
        tokenizer.save_pretrained('output/best')
        print(f"   💾 Saved best model (acc={val_acc:.1%})")

print(f"\n🎉 Training complete! Best accuracy: {best_val_acc:.1%}")

In [ ]:
# Cell 8: Save final model
import shutil
import json

# Save final model
model.save_pretrained('output/final')
tokenizer.save_pretrained('output/final')

# Copy label map
shutil.copy('label_map.json', 'output/final/label_map.json')

# Save training history
with open('output/final/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print("📁 Saved files:")
!ls -la output/final/

In [ ]:
# Cell 9: Test predictions
id_to_label = {v: k for k, v in label_map.items()}

test_queries = [
    "play some jazz music",
    "what's the weather like",
    "set a timer for 5 minutes",
    "turn off the lights",
    "I'm feeling stressed",
    "remind me to call mom",
    "help me meditate",
]

print("🧪 Testing predictions:")
print("-" * 50)

model.eval()
for query in test_queries:
    inputs = tokenizer(query, return_tensors='pt', truncation=True, max_length=128).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits[0])
        top_k = torch.topk(probs, k=3)

    print(f"\n📝 \"{query}\"")
    for score, idx in zip(top_k.values.tolist(), top_k.indices.tolist()):
        tool = id_to_label.get(idx, f"unknown_{idx}")
        conf = "🟢" if score > 0.5 else "🟡" if score > 0.2 else "⚪"
        print(f"   {conf} {tool}: {score:.1%}")

In [ ]:
# Cell 10: Download model
!cd output && zip -r ../ferni-router-v2.zip final/
files.download('ferni-router-v2.zip')
print("\n✅ Download complete! Extract to models/ferni-router-qwen/")